# DSA 504 — Class 4
## Intro to pandas: Series, DataFrames, and Reading a Real CSV

**Date:** Monday, Sep 14
**Reading:** *Python for Data Analysis*, ch. 5–6

---

### A quick note before we start

Every class so far has used small, illustrative retail-sales examples typed directly into the code. **Today that changes.** We're loading a real file — `retail_sales.csv` — the dataset that will run through the rest of the semester. It has ~4,000 rows across 4 stores and 5 categories over one year.

**Important:** this file is not perfectly clean. You will notice missing values and inconsistent category names (e.g., `"Electronics"` vs `"electronics"` vs `"ELECTRONICS"`) while inspecting it today. **That's intentional — don't fix it yet.** Today is about *seeing* what real data looks like. Classes 6–7 are entirely about cleaning it properly.

---

### Learning goals
By the end of this class, you will be able to:
- Explain the difference between a pandas `Series` and a `DataFrame`
- Create a `Series` and a `DataFrame` from scratch
- Read a real CSV file into a DataFrame with `pd.read_csv()`
- Use the standard first-look inspection tools: `.head()`, `.tail()`, `.shape`, `.columns`, `.dtypes`, `.info()`, `.describe()`
- Spot real data-quality issues (missing values, inconsistent categories, duplicates) using `.isna().sum()`, `.unique()`, and `.duplicated()`


## 1. What Is pandas, and Why Does It Exist?

Up to now, we've represented one row of data as a dictionary, and many rows as a list of dictionaries (Class 3). That works, but it gets slow and clumsy once you have thousands of rows — looping over everything by hand for every single question you want to ask.

**pandas** is a library built specifically for tabular data (rows and columns, like a spreadsheet). It gives you fast, pre-built tools for exactly the kinds of things you'd otherwise have to hand-write loops for.


In [ ]:
import pandas as pd
import numpy as np

print(pd.__version__)


## 2. `Series` — A Single Column of Data

A pandas **Series** is a one-dimensional labeled array — think of it as a single column from a spreadsheet, with an index attached to each value.


In [ ]:
# Creating a Series from a plain Python list
daily_units = pd.Series([138, 95, 210, 60, 175])
print(daily_units)
print(type(daily_units))


Notice the left-hand column — that's the **index**. By default it's just 0, 1, 2, 3... but you can set it to something meaningful, like dates.


In [ ]:
# A Series with a custom index
daily_units = pd.Series(
    [138, 95, 210, 60, 175],
    index=["Mon", "Tue", "Wed", "Thu", "Fri"]
)
print(daily_units)

# Now you can access a value by its meaningful label
print(daily_units["Wed"])


In [ ]:
# Series support the same kind of operations you'd expect from a list, plus more
daily_units = pd.Series([138, 95, 210, 60, 175])

print(daily_units.sum())     # 678
print(daily_units.mean())    # average
print(daily_units.max())     # highest value
print(daily_units > 100)     # returns a Series of True/False -- one per value


## 3. `DataFrame` — A Full Table

A **DataFrame** is the pandas structure for a full table: multiple columns (each one technically a `Series`), sharing a common index. This is the structure you'll use constantly for the rest of the semester.


In [ ]:
# Creating a small DataFrame from a dictionary of lists
# (this is the same shape of data as the list-of-dictionaries pattern from Class 3)
data = {
    "store": ["Utica", "Albany", "Rome"],
    "units_sold": [138, 95, 210],
    "revenue": [4521.75, 3980.10, 5102.00]
}

sales_df = pd.DataFrame(data)
print(sales_df)
print(type(sales_df))


**Connecting this to Class 3:** notice the dictionary keys became column names, and each list became a column of values. This is the exact same information as a list of `{"store": ..., "units_sold": ..., "revenue": ...}` dictionaries — just organized as columns instead of a list of rows. pandas can build a DataFrame from either shape.


In [ ]:
# You can also build a DataFrame from a list of dictionaries -- same result
records = [
    {"store": "Utica",  "units_sold": 138, "revenue": 4521.75},
    {"store": "Albany", "units_sold": 95,  "revenue": 3980.10},
    {"store": "Rome",   "units_sold": 210, "revenue": 5102.00},
]

sales_df_v2 = pd.DataFrame(records)
print(sales_df_v2)


In [ ]:
# Accessing a single column returns a Series
print(sales_df["revenue"])
print(type(sales_df["revenue"]))

# Accessing a single row by position uses .iloc[]
print(sales_df.iloc[0])


## 4. Reading a Real CSV File

Now for the real dataset. Make sure `retail_sales.csv` is in the same folder as this notebook.


In [ ]:
sales = pd.read_csv("retail_sales.csv")
print(type(sales))
print(f"Loaded {len(sales)} rows")


That's it — one line reads the entire file into a DataFrame, regardless of whether it has 5 rows or 5 million.


## 5. Basic Inspection — Getting Your Bearings on New Data

Whenever you load a dataset for the first time, run through the same handful of checks before doing anything else. This becomes muscle memory.


In [ ]:
# .head() shows the first 5 rows by default -- your first look at the data
sales.head()


In [ ]:
# .head(n) lets you control how many rows
sales.head(10)


In [ ]:
# .tail() shows the LAST rows -- useful for checking the end of a file loaded correctly
sales.tail()


In [ ]:
# .shape tells you (rows, columns)
print(sales.shape)


In [ ]:
# .columns lists the column names
print(sales.columns)


In [ ]:
# .dtypes shows the data type pandas inferred for each column
print(sales.dtypes)


**Talking point:** notice `date` is read in as an `object` (pandas' term for text/string), not an actual date. This is extremely common, and it's exactly the kind of thing `.dtypes` is for catching. We'll fix this properly in Classes 6–7.


In [ ]:
# .info() gives you a compact summary: dtypes, non-null counts, and memory usage all at once
sales.info()


**Look closely at the "Non-Null Count" column above.** If a column's non-null count is *less* than the total row count, that column has missing values. This is your first real signal that `units_sold` and `revenue` aren't fully clean.


In [ ]:
# .describe() gives summary statistics for numeric columns: count, mean, std, min, max, quartiles
sales.describe()


## 6. Spotting Data Quality Issues (Observing Only — Not Fixing Yet)

Real datasets are messy. Today we just learn how to *see* the mess. Classes 6–7 teach you how to fix it.


In [ ]:
# Counting missing values per column
print(sales.isna().sum())


In [ ]:
# .unique() shows every distinct value in a column -- useful for spotting inconsistent text
print(sorted(sales["category"].unique()))


**Look at that list carefully.** You should see things like `"Electronics"`, `"electronics"`, and `"ELECTRONICS"` all listed as if they were different categories — even though they clearly represent the same thing. This is one of the most common real-world data problems, and it's exactly why `.unique()` is one of the first things you check on any new categorical column.


In [ ]:
# .nunique() just gives you the COUNT of unique values, not the values themselves
print(sales["category"].nunique())
# Notice this number is larger than 5 -- even though there are only 5 real categories


In [ ]:
# Checking for duplicate rows
print("Number of exact duplicate rows:", sales.duplicated().sum())


**Why this matters:** if you calculated total revenue right now without addressing duplicates and inconsistent categories, your numbers would be wrong — duplicated rows would inflate totals, and `"Electronics"` vs `"electronics"` would be counted as two separate categories instead of one. This is exactly why cleaning (Classes 6–7) has to happen before serious analysis, not after.


---
## Guided Practice

Work through these using `retail_sales.csv`, already loaded above as `sales`.


### Exercise 1 — First look
Print the shape of the dataset, the column names, and the data types of each column.


In [ ]:
# Exercise 1 — your code here



### Exercise 2 — Missing values
Print how many missing values exist in each column. Which two columns have missing values, and roughly what fraction of the total rows does that represent?


In [ ]:
# Exercise 2 — your code here



### Exercise 3 — Unique stores and categories
Print the unique values in the `store` column, and separately the unique values in the `category` column. How many *real* categories do you think there actually are, based on what you see?


In [ ]:
# Exercise 3 — your code here



### Exercise 4 (stretch) — Summary statistics by eye
Using `.describe()`, answer: what is the maximum value of `units_sold` in the dataset, and does that seem like a realistic number given what you know about the data (4 stores, 5 categories, roughly daily records)?


In [ ]:
# Exercise 4 — your code here



---
## Solutions

Try each exercise yourself first. These are here for after class, or once you're stuck.


---
## Wrap-up

**Recap:** the difference between a `Series` (one column) and a `DataFrame` (a full table); reading a real CSV with `pd.read_csv()`; the standard first-look toolkit (`.head()`, `.tail()`, `.shape`, `.columns`, `.dtypes`, `.info()`, `.describe()`); and spotting (not yet fixing) missing values, inconsistent categories, and duplicates.

**Common mistakes to watch for:**
- Forgetting that `retail_sales.csv` must be in the same folder as your notebook, or using the wrong file path
- Confusing `.shape` (no parentheses — it's an attribute) with `.describe()` (a method — needs parentheses)
- Assuming a column's dtype is correct just because it loaded without an error — always check `.dtypes` explicitly
- Treating `.unique()` output as "these are all different real categories" instead of noticing formatting inconsistencies

**Before next class (Sep 16):** Class 5 is pandas select/filter/sort/group — you'll start actually asking questions of this same dataset (e.g., "what's total revenue per store?"), which will make the messiness we spotted today even more obviously a problem worth fixing soon.
